Week 2

Step 1: Load the Data

In [1]:
import pandas as pd

In [2]:
listed_data = pd.read_csv('listed.csv')

/var/folders/mf/9m6y4nlx41z0kjyljz4sn3tc0000gn/T/ipykernel_3701/4097304770.py:1: DtypeWarning: Columns (0: ListAgentEmail, 1: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  listed_data = pd.read_csv('listed.csv')


Row count before Residential Filter (includes ALL PropertyType's): 853331 

Row count AFTER Residential filter: 540341


In [6]:
listed_data.shape

(540341, 84)

Missing Value Analysis

In [7]:
missing_counts = listed_data.isna().sum()

missing_pct = (missing_counts / len(listed_data)) * 100

missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_pct": missing_pct
})

missing_summary["90pct_missing"] = missing_summary["missing_pct"] > 90

missing_summary = missing_summary.sort_values(by="missing_pct", ascending=False)

missing_summary.head(5)

,missing_count,missing_pct,90pct_missing
TaxAnnualAmount,540341,100.0,True
FireplacesTotal,540341,100.0,True
ElementarySchoolDistrict,540341,100.0,True
BusinessType,540341,100.0,True
TaxYear,540341,100.0,True


Numeric Distribution Review

In [8]:
cols = [
    "ClosePrice", "ListPrice", "OriginalListPrice",
    "LivingArea", "LotSizeAcres",
    "BedroomsTotal", "BathroomsTotalInteger",
    "DaysOnMarket", "YearBuilt"
]

Summary Statistics

In [10]:
summary_list = []
for col in cols: 
    data = listed_data[col].dropna()
    summary = {
        "column": col,
        "count": len(data),
        "mean": data.mean(),
        "median": data.median(),
        "std": data.std(),
        "min": data.min(),
        "max": data.max(),
    }
    
    summary_list.append(summary)
    
summary_df = pd.DataFrame(summary_list)
summary_df.head(5)

,column,count,mean,median,std,min,max
0,ClosePrice,142967,1.201249e+06,851000.00,4.329322e+06,525.0,8.200000e+08
1,ListPrice,540341,1.314106e+06,845000.00,2.348922e+06,100.0,1.950000e+08
2,OriginalListPrice,539574,1.398075e+06,849000.00,7.356544e+06,0.0,1.390000e+09
3,LivingArea,539789,1.980076e+03,1669.00,2.337920e+04,0.0,1.702132e+07
4,LotSizeAcres,495766,6.525178e+01,0.17,1.213560e+04,0.0,4.187292e+06


Percentile Statistics

In [12]:
percentiles = data.quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
column_percentile = []
outlier_dict = {}

for col in cols:
    data = listed_data[col].dropna()
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = data[(data < lower_bound) | (data > upper_bound)]
    
    outlier_dict[col] = outliers

    percentages = {
        "column": col,
        "1%": percentiles.loc[0.01],
        "99%": percentiles.loc[0.99],
        "outlier_count": len(outliers)
    }
    
    column_percentile.append(percentages)
    column_distribution = pd.DataFrame(column_percentile)
    
column_distribution.head()


,column,1%,99%,outlier_count
0,ClosePrice,1911.0,2025.0,10252
1,ListPrice,1911.0,2025.0,45585
2,OriginalListPrice,1911.0,2025.0,45428
3,LivingArea,1911.0,2025.0,26737
4,LotSizeAcres,1911.0,2025.0,79447
